In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78521)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 56307)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 55495)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 18897)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 18895)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import AttentionSetPooling
from modules.model import MultiLatentModel
from modules.norm import LogCounts
from modules.train import MultiTrainer, MultiTrainerStage, Experiment, grid
from modules.trainers import ReconstrTrainer, ClassifTrainer
from modules.loss import NBLoss, KLDLoss, MultiLoss

import torch.nn as nn
from functools import partial

In [3]:
# multihead model
model = MultiLatentModel(
    dataset = dataset,
    embed_dim = 128,
    # head_dim = None (default)
    # num_heads = 1 (default)
    method = 'set',

    # layers
    norm_class = LogCounts,
    encoder_class = nn.Linear,
    pooling_class = AttentionSetPooling,
    mlp = False,
    variational = True,
    # out_module = nn.Linear (default)

    # layer params
    hidden_dims = 1,
    act_fn = nn.ReLU, 
    norm_fn = 'layer', 
    end_fn = False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True}
    # pooling_kwargs = None (default)
)

In [4]:
# trainer
ae_stage = MultiTrainerStage(
    name = 'ae',
    train = ['encoder', 'latent_ae', 'decoder'],
    trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys=('x','x_pred','theta','z_mu_ae','z_logvar_ae'),
        metric_keys={'pred':'x_pred', 'target':'x'},
        
        loss_class=MultiLoss,
        loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu_ae', 'logvar':'z_logvar_ae'}
        ]
    },
    )
)

cl_stage = MultiTrainerStage(
    name = 'cl',
    train = ['latent_cl', 'classifier'],
    trainer = ClassifTrainer(
        lr=1e-3, 

        early_stop=True,
        stop_metric='accuracy',
        stop_kwargs={'mode':'max'},

        kw_keys={'input':'y_logits', 'target':'y'},
        metric_keys={'pred':'input', 'target':'target'},
        loss_class=nn.CrossEntropyLoss,
        # loss_kwargs={'label_smoothing':0.1}
    )
)

trainer = MultiTrainer(
    stages = [ae_stage, cl_stage]
)

In [5]:
# expt
expt = Experiment(
    num_trials=5, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_config(
    name = 'pathway_vae',
    model = model,
    trainer = trainer,
)

In [6]:
expt.run_experiment(
    'benchmark_2_attn',
    report_metrics=['loss','kld','nb','mae'],
    save_model=True,
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]